# New Tokyo PoC Script

## 1 · Scene & Environment Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use GPU 0, will use CPU otherwise
import numpy as np
import pandas as pd
import json
import csv as _csv
import poc.setup as poc
import poc.utils as utils
import poc.rt as rt
import poc.topology as tp

path_suffix = "/home/polimi"

# Antennas are Car 1: [1, 2], RSU: [30, 31, 40], Car 2: [5, 6, 7]
transmitters = [30, 31, 5, 6]
receivers = [1, 2, 40, 7]

scene = poc.setup_scenario(
    file_name=f"{path_suffix}/tokyo-polimi-dt-jsac/scenarios/ookayama_full_flat/ookayama.xml",
    frequency=60e9,
    bandwidth=1000e6,
    verbose=False,
    time_checker=False
)

poc.setup_coordinate_systems(
    # Using the right corner of the porch of the main building (facing north) as reference point
    ref_sionna_x=162.396, ref_sionna_y=85.782, # Sionna RT
    ref_external_x=-13524.5, ref_external_y=-43817.64 # Tokyo Mobility DT
)

poc.setup_antenna_type(
    transmitters=transmitters,
    receivers=receivers,
    pattern="load_custom", # Use custom pattern defined by CSV files
    elevation_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/elevation_beam_16.csv",
    azimuth_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/azimuth_beam_16.csv",
    polarization="H", # Use horizontal (define how is mounted in setup_antenna_on_object)
    simulate_perfect_beamforming=True,
    use_look_at_ideal_pointing=False, # If False, use the custom point_toward_peer() function that considers the actual antenna orientation and heading, which is more realistic but may not achieve perfect alignment
    beam_sweeping_angle=60 # [-60, 60]
)

# Peers (Tx)->(Rx) are 30->1, 31->7, 6->40, 5->2
poc.setup_wireless_links(
    wireless_links=[(30, 1), (31, 7), (6, 40), (5, 2)] # (Tx, Rx)
)

poc.setup_rt(
    rt_max_depth=3,
    rt_los=True,
    rt_specular_reflection=True,
    rt_diffuse_reflection=True,
    rt_refraction=False,
    rt_diffraction=False,
    rt_corner_diffraction=False,
    rt_max_num_paths_per_src=1e5,
    rt_samples_per_src=1e5
)

poc.setup_filters(
    transmitters=transmitters,
    receivers=receivers,
    use_moving_average_filter=True,
    moving_average_window_size=5
)

struc = poc.startup(port=35944)

    [INFO] Loaded scene with frequency 60.0 GHz, bandwidth 1000.0 MHz.
    [INFO] Expecting UDP messages from Tokyo Digital Twin on UDP/35944
    [INFO] Setup procedure complete.


## 2 · Object configurator

In [2]:
# ── Asset paths & TX powers ───────────────────────────────────────────────────
CAR_1_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/coms.ply"
CAR_2_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/estima.ply"
RSU_MESH         = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/rsu.ply"
TREE_MESH        = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/tree.ply"


# Add objects to the scene
poc.add_object(ref_obj_id=1, mesh_path=CAR_1_MESH, position=[193.263, 182.911, 0.758495])
poc.add_object(ref_obj_id=2, mesh_path=CAR_2_MESH, position=[213.124, 191.882, 0.874555])
poc.add_object(ref_obj_id=101, mesh_path=RSU_MESH, position=[192.407, 195.791, 5.0/2])

trees = {}
zt = 8.64/2
trees[1] = [189.811, 185.500, zt]
trees[2] = [190.055, 184.113, zt]
trees[3] = [189.510, 180.790, zt]
trees[4] = [188.870, 177.861, zt]
trees[5] = [188.285, 173.964, zt]
trees[6] = [188.032, 170.855, zt]
trees[7] = [187.597, 167.868, zt]
trees[8] = [187.121, 164.304, zt]
trees[9] = [186.578, 161.490, zt]
trees[10] = [185.993, 158.196, zt]
trees[11] = [185.631, 155.328, zt]
trees[12] = [185.161, 152.369, zt]

for i in range(1, len(trees) + 1):
    poc.add_tree(ref_tree_id=i, mesh_path=TREE_MESH, position=trees[i])

# Add antennas to each object
# Car 1 (Antennas: 1, 2)
poc.mount_antenna_on_object(ref_obj_id=1, 
                            ant_id=1, # peer=30 - V2I Client (Rx)
                            displacement=[-0.359, -0.048, 0.837505], orientation=[0, 0, 0], mounted_vertically=False)
poc.mount_antenna_on_object(ref_obj_id=1,  
                            ant_id=2, # peer=5 - V2V Client (Rx)
                            displacement=[0.719, -0.027, 0.319505], orientation=np.array([0, np.radians(-5), 0]), mounted_vertically=False)

# RSU (Antennas: 30, 31, 40)
poc.mount_antenna_on_object(ref_obj_id=101,
                            ant_id=30, # peer=1 - I2V Server (Tx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(220), np.radians(25), 0]), mounted_vertically=False,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=101, 
                            ant_id=31, # peer=7 - I2V Server (Tx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(220), np.radians(25), 0]), mounted_vertically=False,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=101, 
                            ant_id=40, # peer=6 - I2V Client (Rx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(-45), np.radians(25), 0]), mounted_vertically=False)

# Car 2 (Antennas: 5, 6, 7)
poc.mount_antenna_on_object(ref_obj_id=2, 
                            ant_id=5, #peer=2 - V2V Server (Tx)
                            displacement=[-1.024, -0.048, 1.071445], orientation=np.array([np.pi, np.radians(-10), 0]), mounted_vertically=False,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=2, 
                            ant_id=6, # peer=40 - V2I Server (Tx)
                            displacement=[-0.431, -0.009, 1.071445], orientation=[np.pi, 0, 0], mounted_vertically=False,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=2, 
                            ant_id=7, # peer=31 - V2I Client (Rx)
                            displacement=[-0.34, -0.037, 1.072445], orientation=np.array([0, np.radians(-10), 0]), mounted_vertically=False)

     [INFO] Found peer antenna for Rx 1 in wireless_links: its Tx is 30. Perfect beamforming simulation will be applied between these two antennas.
     [INFO] Found peer antenna for Rx 2 in wireless_links: its Tx is 5. Perfect beamforming simulation will be applied between these two antennas.
     [INFO] Found peer antenna for Tx 30 in wireless_links: its Rx is 1. Perfect beamforming simulation will be applied between these two antennas.
     [INFO] Found peer antenna for Tx 31 in wireless_links: its Rx is 7. Perfect beamforming simulation will be applied between these two antennas.
     [INFO] Found peer antenna for Rx 40 in wireless_links: its Tx is 6. Perfect beamforming simulation will be applied between these two antennas.
     [INFO] Found peer antenna for Tx 5 in wireless_links: its Rx is 2. Perfect beamforming simulation will be applied between these two antennas.
     [INFO] Found peer antenna for Tx 6 in wireless_links: its Rx is 40. Perfect beamforming simulation will be ap

## 3 · Showtime!

In [ ]:
import time
import json
import csv
import poc.rt as rt
import poc.utils as utils

udp_socket = struc["udp_socket"]
verbose = True
horizon = 1

def main():
    global horizon
    timestamp = None
    
    print("Starting main loop...")

    while True:

        # - - - - - - - - - - - - - - - - - - - - -

        # 1. RECEIVE AND PARSE MESSAGE FROM SOCKET

        # - - - - - - - - - - - - - - - - - - - - -

        # Receive data from the socket
        payload, address = udp_socket.recvfrom(1024*10)
        struc["latest_msg_address"] = address
        message = payload.decode()

        print(f"\nReceived message from {address}: {message}")
        
        t_tot = time.time()
        # Parse the JSON message
        data = json.loads(message)

        # Check message type early and route accordingly
        if isinstance(data, dict) and "positions" in data and "links" in data:
            # Position/Link update message
            timestamp = data["time"]
            positions = data["positions"]
            links = data["links"]

        # Check if it is a configuration message 
        msg_entries = data.get("data") if isinstance(data.get("data"), list) else (data if isinstance(data, list) else [data])
        type_val = msg_entries[0].get("type") if msg_entries and isinstance(msg_entries[0], dict) else None

        # Apply RT_CONFIGURATION message
        if type_val and "RT_CONFIGURATION" in type_val.upper():
            utils.manage_online_reconfiguration(msg_entries, struc, is_manual_override=False)
            # Logs control message
            local_timestamp = time.time()
            with open(struc["log_file"], mode="a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow([local_timestamp, timestamp, None,
                                message, None, None, None, None, None, None, None, None, None, None, None, None, None])

            continue

        # Apply CHANGE_PREDICTION_HORIZON message
        if type_val and "CHANGE_PREDICTION_HORIZON" in type_val.upper():
            horizon = float(msg_entries[0].get('value'))
            # Logs control message
            local_timestamp = time.time()
            with open(struc["log_file"], mode="a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow([local_timestamp, timestamp, None,
                                message, None, None, None, None, None, None, None, None, None, None, None, None, None])
            
            continue


        if verbose:
            print(" ")
            print("  - - - - - - - - - - - - - - - - -  NEW PREDICTION REQUEST  - - - - - - - - - - - - - - - - -  ")
            print(" ")



        
        # - - - - - - - - - - - - - - - - - - - - -

        # 2. EXTRACT MEASURED RSSIs FOR CALIBRATION

        # - - - - - - - - - - - - - - - - - - - - -

        if 'is_stale' in links["vehicle1.lower_sta__vehicle3.back_horizontal_ap"]:
            current_5vs2 = links["vehicle1.lower_sta__vehicle3.back_horizontal_ap"]["rssi_dbm"] if links["vehicle1.lower_sta__vehicle3.back_horizontal_ap"]["is_stale"] == False else None
        else:
            print("No measurement for 5vs2")
            current_5vs2 = None

        if 'is_stale' in links["rsu.south_relay__vehicle1.upper_sta"]:    
            current_30vs1 = links["rsu.south_relay__vehicle1.upper_sta"]["rssi_dbm"] if links["rsu.south_relay__vehicle1.upper_sta"]["is_stale"] == False else None
        else:
            print("No measurement for 30vs1")
            current_30vs1 = None

        if 'is_stale' in links["rsu.south_relay__vehicle3.front_upper_sta"]:
            current_31vs7 = links["rsu.south_relay__vehicle3.front_upper_sta"]["rssi_dbm"] if links["rsu.south_relay__vehicle3.front_upper_sta"]["is_stale"] == False else None
        else:
            print("No measurement for 31vs7")
            current_31vs7 = None

        if 'is_stale' in links["rsu.east_sta__vehicle3.back_upper_ap"]:    
            current_6vs40 = links["rsu.east_sta__vehicle3.back_upper_ap"]["rssi_dbm"] if links["rsu.east_sta__vehicle3.back_upper_ap"]["is_stale"] == False else None
        else:
            print("No measurement for 6vs40")
            current_6vs40 = None


        # - - - - - - - - - - - - - - - - - - - - -

        # 3. APPLY PREDICTED LOCATIONS IN SIONNA

        # - - - - - - - - - - - - - - - - - - - - -

        # Update the locations in the scenario
        t = time.time()

        for vehicle_id, trajectory in positions.items():
            for point in trajectory:
                dt = point['dt']
                if dt == horizon:
                    x = point['x']
                    y = point['y']
                    z = point['z'] 
                    yaw = -point['yaw'] + 90

                    if vehicle_id == '1':
                        # Update Car 1's position in Sionna
                        utils.move_object(ref_obj_id=1, 
                                            position=[x, z, 0.758495], 
                                            ref_system="external",
                                            heading_angle=yaw, 
                                            velocity=5,
                                            sionna_structure=struc)
                        print(f"Updating Vehicle {vehicle_id} at dt={dt:.3f}s to position (x={x:.2f}, y={y:.2f}, z={z:.2f}) with yaw={yaw:.2f}°")
                    
                    elif vehicle_id == '3':
                        # Update Car 2's position in Sionna
                        utils.move_object(ref_obj_id=2, 
                                            position=[x, z, 0.975445], 
                                            ref_system="external",
                                            heading_angle=yaw, 
                                            velocity=5,
                                            sionna_structure=struc)
                        print(f"Updating Vehicle {vehicle_id} at dt={dt:.3f}s to position (x={x:.2f}, y={y:.2f}, z={z:.2f}) with yaw={yaw:.2f}°")
        
        location_update_time = (time.time() - t) * 1000
        if struc["time_checker"]:
            print(f"        Location update took: {location_update_time:.2f} ms")



        # - - - - - - - - - - - - - - - - - - - - -

        # 4. COMPUTE FUTURE RSSI AND NET TOPOLOGY

        # - - - - - - - - - - - - - - - - - - - - -
       
        # Compute RSSIs
        t_c = time.time()
        raw_rssi_5vs2 = rt.compute_rssi(5, 2, struc)
        rssi_5vs2 = struc["filters"][('5', '2')].step(raw_rssi_5vs2, current_5vs2)
        los_5vs2 = rt.compute_los_status(5, 2, struc)
        can_bf_5vs2 = utils.can_beamform(5, 2, struc)

        raw_rssi_6vs40 = rt.compute_rssi(6, 40, struc)
        rssi_6vs40 = struc["filters"][('6', '40')].step(raw_rssi_6vs40, current_6vs40)
        los_6vs40 = rt.compute_los_status(6, 40, struc)
        can_bf_6vs40 = utils.can_beamform(6, 40, struc)

        raw_rssi_30vs1 = rt.compute_rssi(30, 1, struc)
        rssi_30vs1 = struc["filters"][('30', '1')].step(raw_rssi_30vs1, current_30vs1)
        los_30vs1 = rt.compute_los_status(30, 1, struc)
        can_bf_30vs1 = utils.can_beamform(30, 1, struc)

        raw_rssi_31vs7 = rt.compute_rssi(31, 7, struc)
        rssi_31vs7 = struc["filters"][('31', '7')].step(raw_rssi_31vs7, current_31vs7)
        los_31vs7 = rt.compute_los_status(31, 7, struc)
        can_bf_31vs7 = utils.can_beamform(31, 7, struc)

        # Compute topology (change the margin if necessary)
        topology = tp.evaluate_best_topology(sionna_structure=struc, margin_dbm=5,
                                             rssi_5vs2=rssi_5vs2, 
                                             rssi_6vs40=rssi_6vs40, 
                                             rssi_30vs1=rssi_30vs1, 
                                             rssi_31vs7=rssi_31vs7)


        prediction_time = (time.time() - t_c) * 1000
        if struc["time_checker"]:
            print(f"        RT and Topology Prediction took: {prediction_time:.2f} ms")



        # - - - - - - - - - - - - - - - - - - - - -

        # 5. SEND BACK FUTURE NETWORK TOPOLOGY

        # - - - - - - - - - - - - - - - - - - - - -

        # Prepare the response
        response = [{}]
        response[0]["horizon"] = horizon
        response[0]["request_time"] = timestamp
        response[0]["data"] = {}
        response[0]["data"]["topology"] = topology

        response = json.dumps(response, default=lambda o: float(o) if isinstance(o, np.float32) else o)

        # Send back the response
        udp_socket.sendto(response.encode(), ("11.0.0.1", 5560))
        #udp_socket.sendto(response.encode(), address) # Use this for local testing script
        total_elapsed_time = (time.time() - t_tot) * 1000

        if verbose:
            print(" ")
            print(f"  - - - - - - - - - - - - - - - - -   Handled in {total_elapsed_time:.2f} ms   - - - - - - - - - - - - - - - - -  ")
            print(" ")
            print(" ")
        
        # Logging
        local_unix_timestamp = time.time()
        with open(struc["log_file"], mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([local_unix_timestamp, timestamp, horizon,
                             message, response,
                             struc["sionna_location_db"][1]['x'], struc["sionna_location_db"][1]['y'], struc["sionna_location_db"][1]['angle'],
                             struc["sionna_location_db"][2]['x'], struc["sionna_location_db"][2]['y'], struc["sionna_location_db"][2]['angle'],
                             raw_rssi_5vs2, raw_rssi_6vs40, raw_rssi_30vs1, raw_rssi_31vs7,
                             rssi_5vs2, rssi_6vs40, rssi_30vs1, rssi_31vs7,
                             los_5vs2, los_6vs40, los_30vs1, los_31vs7,
                             can_bf_5vs2, can_bf_6vs40, can_bf_30vs1, can_bf_31vs7,
                             location_update_time, prediction_time, total_elapsed_time])
        
        '''
        from sionna.rt import render_to_file
        # Frame-by-frame exporter
        global frame_num
        print("Exporting frame ", frame_num)
        
        struc["scene"].render_to_file(camera=my_cam, 
                                         filename=f"/home/rpegurri/Tokyo NDT Integration/PoC Tokyo Science/export/frame_{frame_num}.png", 
                                         show_devices=False)
        frame_num += 1
        '''
        

        if message.startswith("SHUTDOWN_SIONNA"):
            print("Got SHUTDOWN_SIONNA message. Bye!")
            udp_socket.close()
            break

# Entry point
if __name__ == "__main__":
    main()


Starting main loop...

Received message from ('11.0.0.1', 60736): {"time": 1151.920410156, "positions": {"1": [{"dt": 0.0, "x": -13489.28033202624, "y": -43708.76849666227, "z": 70.0, "yaw": 14.896902673401405}, {"dt": 0.125, "x": -13488.7275390625, "y": 70.0, "z": -43708.9296875, "yaw": -37.300025939941406}, {"dt": 0.25, "x": -13488.09765625, "y": 70.0, "z": -43709.23046875, "yaw": -33.22018814086914}, {"dt": 0.375, "x": -13487.8974609375, "y": 70.0, "z": -43709.80859375, "yaw": -27.106904983520508}, {"dt": 0.5, "x": -13487.58203125, "y": 70.0, "z": -43710.04296875, "yaw": -25.207317352294922}, {"dt": 0.625, "x": -13487.2509765625, "y": 70.0, "z": -43710.125, "yaw": -24.668027877807617}, {"dt": 0.75, "x": -13486.9609375, "y": 70.0, "z": -43710.1796875, "yaw": -25.19873809814453}, {"dt": 0.875, "x": -13486.7265625, "y": 70.0, "z": -43710.23828125, "yaw": -26.100915908813477}, {"dt": 1.0, "x": -13486.5185546875, "y": 70.0, "z": -43710.296875, "yaw": -26.655712127685547}, {"dt": 1.125, "

KeyboardInterrupt: 

# Utilities

### Scene previewer

In [ ]:
scene.preview(resolution=(1000, 1000), show_orientations=True, show_devices=True)

### Mobility quick tests

In [ ]:
utils.move_object(ref_obj_id=1, 
                  position=[195.584, 189.436, 0.758495],
                  ref_system="sionna",
                  heading_angle=0, 
                  velocity=5,
                  sionna_structure=struc)

utils.move_object(ref_obj_id=2, 
                  position=[204.080, 188.896, 0.975445], 
                  ref_system="sionna",
                  heading_angle=190, 
                  velocity=2,
                  sionna_structure=struc)

In [ ]:
a1 = -14.9 
a2 = -6.88

a1a = -a1 + 90
a2a = -a2 + 90 

utils.move_object(ref_obj_id=1, 
                  position=[-13490.44, -43708.46, 0.758495],
                  heading_angle=a1a, 
                  velocity=5,
                  sionna_structure=struc)

utils.move_object(ref_obj_id=2, 
                  position=[-13485.85, -43709.28, 0.975445],
                  heading_angle=a2a, 
                  velocity=5,
                  sionna_structure=struc)

In [ ]:
utils.move_object(ref_obj_id=2, 
                  position=[-13484.683, -43709.469, 0.975445],
                  heading_angle=-7-90, 
                  velocity=5,
                  sionna_structure=struc)

### Topology computation

In [ ]:
topology = tp.evaluate_best_topology(sionna_structure=struc)
topology

### Displacement calculator
Note: displacement is computed with cars at their default orientation after being imported (facing Ookayama north)

In [ ]:
car_1 = [193.263, 182.911, 0.758495]
car_2 = [213.124, 191.882, 0.874555]

car_1 = scene.get("obj_1").position
car_2 = scene.get("obj_2").position

car_1 = [car_1[0][0], car_1[1][0], car_1[2][0]]
car_2 = [car_2[0][0], car_2[1][0], car_2[2][0]]
print("Car 1 position:", car_1)
print("Car 2 position:", car_2)

# Car 1
ant_1 = [192.904, 182.863, 1.496 + 0.1]
ant_2 = [193.982, 182.884, 0.978 + 0.1]

# Car 2
ant_5 = [212.100, 191.834, 1.746 + 0.20]
ant_6 = [212.693, 191.873, 1.746 + 0.20]
ant_7 = [212.784, 191.845, 1.747 + 0.20]

displacement_1 = np.array(ant_1) - np.array(car_1)
displacement_2 = np.array(ant_2) - np.array(car_1)
displacement_5 = np.array(ant_5) - np.array(car_2)
displacement_6 = np.array(ant_6) - np.array(car_2)
displacement_7 = np.array(ant_7) - np.array(car_2)

print("Displacement Antenna 1:", displacement_1)
print("Displacement Antenna 2:", displacement_2)
print("Displacement Antenna 5:", displacement_5)
print("Displacement Antenna 6:", displacement_6)
print("Displacement Antenna 7:", displacement_7)